In [0]:
from pyspark.sql.functions import *


In [0]:
df = (
    spark.read.table("marketing_analytics_project.`01_bronze`.conversions")
    .withColumn("date", to_date(col('date')))
    .withColumn("conversion", from_json(
        col('conversion'), "ARRAY<STRUCT<customer_id: STRING, conversion_type : STRING, revenue_usd: DOUBLE>>"
    ))
    .withColumn('conversion', explode(col('conversion')))
    .withColumn("revenue_usd", col("conversion.revenue_usd"))
    .withColumn("conversion_type", col("conversion.conversion_type"))
    .withColumn("customer_id", col("conversion.customer_id"))
    .drop('conversion')
    .select(
            "campaign_id", 
            "customer_id", 
            "date", 
            "conversion_type", 
            "revenue_usd"
        )
        .dropDuplicates(['campaign_id', 'customer_id', 'date', 'conversion_type'])
)

df_2 = (
    df
        .withColumn('add_to_cart', when(col('conversion_type') == 'Add to Cart', 1).otherwise(0))
        .withColumn('wishlist', when(col('conversion_type') == 'Wishlist', 1).otherwise(0))
        .withColumn('purchase', when(col('conversion_type') == 'Buy', 1).otherwise(0))
        .groupBy(['campaign_id', 'date', "customer_id"])
        .agg(
            sum("revenue_usd").alias("revenue_usd"),
            sum("add_to_cart").alias("add_to_cart"),
            sum("wishlist").alias("wishlist"),
            sum("purchase").alias("purchase")
        )
        .orderBy('date')
        
)

display(df_2)

campaign_id,date,customer_id,revenue_usd,add_to_cart,wishlist,purchase
CMP001,2024-06-03,CU523,129.0,0,0,1
CMP001,2024-06-03,CU503,0.0,0,0,0
CMP001,2024-06-04,CU514,0.0,1,0,0
CMP001,2024-06-04,CU513,0.0,1,0,0
CMP001,2024-06-04,CU508,79.99,0,0,1
CMP001,2024-06-05,CU520,0.0,0,1,0
CMP001,2024-06-05,CU509,0.0,1,0,0
CMP001,2024-06-05,CU503,0.0,1,0,0
CMP001,2024-06-08,CU514,0.0,0,1,0
CMP001,2024-06-08,CU520,0.0,1,0,0
